# UC-EDR-1 — Position and Area Queries via OGC API - EDR

**Persona:** Agro-climatologist

**Goal:** Create a demo raster collection via OGC Features transactions, then issue
EDR `position` and `area` queries against it to retrieve environmental data at a
specific geographic point and over a bounding polygon, finish with a deep link to
the EDR browser.

**Prerequisites:**
- GeoID platform running at `DYNASTORE_BASE_URL` (default `http://localhost:8080`)
- `edr` and `features` extensions enabled in the active SCOPE
- Write-scoped JWT in `DYNASTORE_TOKEN` (or IDP client credentials set)
- GDAL raster back-end active for EDR pixel extraction (optional — notebook degrades gracefully)

**References:**
- OGC API — EDR 1.1 (OGC 19-086r5)
- Routes: `/edr/catalogs/{cat}/collections/{col}/position` and `/area`

In [ ]:
import json
import os
import time

import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")

# Auto-provision token via IDP client_credentials when not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass

TOKEN = (
    os.environ.get("DYNASTORE_TOKEN")
    or os.environ.get("DYNASTORE_WRITE_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)

CATALOG_ID    = os.environ.get("CATALOG_ID", "demo-edr")
COLLECTION_ID = os.environ.get("COLLECTION_ID", "ndvi-dekad")

headers = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60.0)

print(f"Platform  : {BASE_URL}")
print(f"Catalog   : {CATALOG_ID}")
print(f"Collection: {COLLECTION_ID}")

## Step 1 — Create demo catalog and collection via OGC Features

The EDR extension queries items that already exist as OGC Features. We create a
lightweight demo catalog + collection with one raster item that carries
`raster:bands` metadata so EDR can build a CoverageJSON response.

If the catalog already exists (409), we skip creation and continue.

In [ ]:
# --- catalog ---
r = client.post(
    "/features/catalogs",
    content=json.dumps({"id": CATALOG_ID, "title": "EDR demo catalog"}),
)
if r.status_code == 409:
    print(f"Catalog '{CATALOG_ID}' already exists — continuing.")
elif r.status_code in (200, 201):
    print(f"Catalog created: {r.json().get('id')}")
else:
    print(f"Catalog creation: {r.status_code} {r.text[:300]}")

# --- collection ---
coll_payload = {
    "id": COLLECTION_ID,
    "title": "NDVI dekadal demo",
    "description": "10-day composite NDVI raster (EDR demo dataset)",
}
r = client.post(
    f"/features/catalogs/{CATALOG_ID}/collections",
    content=json.dumps(coll_payload),
)
if r.status_code == 409:
    print(f"Collection '{COLLECTION_ID}' already exists — continuing.")
elif r.status_code in (200, 201):
    print(f"Collection created: {r.json().get('id')}")
else:
    print(f"Collection: {r.status_code} {r.text[:300]}")

## Step 2 — Ingest a demo raster item

We POST a GeoJSON Feature with `raster:bands` metadata. EDR uses `raster:bands`
to build the CoverageJSON parameter list. We pick a bbox covering central Italy
(a representative agro-climate region) with a `datetime` of 2024-01-11.

In [ ]:
ITEM_ID = "ndvi-2024-01-d1"

item_payload = {
    "type": "Feature",
    "id": ITEM_ID,
    "geometry": {
        "type": "Polygon",
        "coordinates": [[
            [10.0, 40.0], [14.0, 40.0], [14.0, 44.0], [10.0, 44.0], [10.0, 40.0]
        ]],
    },
    "properties": {
        "datetime": "2024-01-11T00:00:00Z",
        "title": "NDVI dekad 1 Jan 2024 — Italy",
        "raster:bands": [
            {"name": "ndvi", "description": "Normalized Difference Vegetation Index",
             "unit": "1", "nodata": -9999, "data_type": "float32",
             "statistics": {"minimum": -0.1, "maximum": 0.95, "mean": 0.45}},
        ],
    },
}

r = client.post(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    content=json.dumps(item_payload),
)
print(f"Item POST: {r.status_code}")
if r.status_code in (200, 201):
    item = r.json()
    print(f"  id       : {item.get('id')}")
    print(f"  datetime : {item.get('properties', {}).get('datetime')}")
elif r.status_code in (409, 422):
    print("  Item already exists or rejected — continuing with existing data.")
else:
    print(f"  {r.text[:300]}")

## Step 3 — List EDR collections

`GET /edr/catalogs/{catalog_id}/collections` returns collections available for
EDR queries, enriched with `parameter_names` derived from `raster:bands`.

In [ ]:
r = client.get(f"/edr/catalogs/{CATALOG_ID}/collections")
print(f"status: {r.status_code}")
if r.status_code == 200:
    colls = r.json().get("collections", [])
    print(f"Collections found: {len(colls)}")
    for c in colls:
        print(f"  id={c.get('id')}  parameters={c.get('parameter_names', [])}")
elif r.status_code == 404:
    print("SKIP: catalog not found — run steps 1-2 first.")
else:
    print(r.text[:300])

## Step 4 — EDR position query

`GET /edr/catalogs/{cat}/collections/{col}/position?coords=POINT(lon lat)&f=covjson`
returns a CoverageJSON Coverage with a scalar domain at the requested location.

The EDR extension reads the nearest raster item for the given datetime and
extracts pixel values at the WKT point. When GDAL is not active (no raster backend),
the server returns 422 or 503 — the notebook prints a SKIP message.

In [ ]:
# Query point: Foligno, Umbria — central Italy agro zone
POINT_LON, POINT_LAT = 12.70, 42.95

r = client.get(
    f"/edr/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/position",
    params={
        "coords": f"POINT({POINT_LON} {POINT_LAT})",
        "datetime": "2024-01-11T00:00:00Z",
        "f": "covjson",
    },
)
print(f"position query status: {r.status_code}")

if r.status_code == 200:
    cov = r.json()
    print(f"type          : {cov.get('type')}")
    domain = cov.get("domain", {})
    axes = domain.get("axes", {})
    print(f"domain.axes.x : {axes.get('x', {}).get('values')}")
    print(f"domain.axes.y : {axes.get('y', {}).get('values')}")
    print(f"domain.axes.t : {axes.get('t', {}).get('values')}")
    print(f"parameters    : {list(cov.get('parameters', {}).keys())}")
    ranges = cov.get("ranges", {})
    for pname, rng in ranges.items():
        print(f"  {pname}: {rng.get('values')}")
elif r.status_code in (404, 422, 503):
    print(f"SKIP: {r.status_code} — GDAL raster backend may not be active or item not found.")
    print(r.text[:300])
else:
    print(r.text[:300])

## Step 5 — EDR area query

`GET /edr/catalogs/{cat}/collections/{col}/area?coords=POLYGON(...)&f=geojson`
returns a GeoJSON FeatureCollection (or CoverageJSON Collection) for the items
whose extents intersect the query polygon.

In [ ]:
# Small polygon around Foligno (central Italy)
AREA_WKT = "POLYGON((12.5 42.8, 13.0 42.8, 13.0 43.1, 12.5 43.1, 12.5 42.8))"

r = client.get(
    f"/edr/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/area",
    params={
        "coords": AREA_WKT,
        "datetime": "2024-01-11T00:00:00Z",
        "f": "geojson",
    },
)
print(f"area query status: {r.status_code}")

if r.status_code == 200:
    result = r.json()
    rtype = result.get("type")
    print(f"response type: {rtype}")
    if rtype == "FeatureCollection":
        feats = result.get("features", [])
        print(f"features: {len(feats)}")
        for f_ in feats[:3]:
            print(f"  id={f_.get('id')}  datetime={f_.get('properties', {}).get('datetime')}")
    else:
        print(json.dumps(result, indent=2)[:500])
elif r.status_code in (404, 422, 503):
    print(f"SKIP: {r.status_code} — GDAL raster backend may not be active or no matching items.")
    print(r.text[:300])
else:
    print(r.text[:300])

## Teardown — Remove demo collection and catalog

Remove the demo data created in steps 1–2. Skip if running against a shared
environment where the collection should remain.

In [ ]:
_TEARDOWN = os.environ.get("EDR_TEARDOWN", "true").lower() == "true"

if _TEARDOWN:
    r = client.delete(f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}")
    print(f"DELETE collection: {r.status_code}")
    r = client.delete(f"/features/catalogs/{CATALOG_ID}")
    print(f"DELETE catalog   : {r.status_code}")
else:
    print("Teardown skipped (set EDR_TEARDOWN=true to enable).")

## Result — Open the EDR browser

In [ ]:
client.close()
print(f"Open the EDR browser: {BASE_URL}/web/pages/edr_browser?language=en")